# Mission 1 — 119 신고자 성별 분류 (M/F) 학습 기록

**최종 모델**: WavLM 3모델 앙상블 25% + ECAPA 로지스틱 회귀 75% (여성 확률 가중 평균, 0.5 이상이면 F)
**서울 Validation 3,640통화**: Accuracy **0.9827** (오류 63) — 공식 추론 명령으로 WAV 전체를 다시 추론한 값

| 구성 | 내용 | 실행 환경 |
| --- | --- | --- |
| Part A | WavLM-base-plus(고정) 임베딩 → Attention+F0 / Stats+F0 / Stats 3모델, 40:40:20 앙상블 | Colab (원본 실행 로그) |
| Part B | ECAPA-TDNN(고정) 화자 임베딩 → 로지스틱 회귀, 내부 holdout으로 섞는 비율 선택 | 로컬 CPU (이 파일에서 실행) |
| 최종 | 공식 명령 1회 실행 추론과 Validation 재추론 검증 | Colab T4 |

**규칙 준수**: 서울 Training만 학습, Validation은 학습에 쓰지 않음. 입력에는 annotation의 `startAt`·`endAt`·`speaker`만
사용(신고자 구간), `gender`는 정답으로만 사용. 공개 사전학습 모델만 사용, 상용 API 없음.
모델·가중치·비율 선택은 Training 내부 holdout(2,920통화)으로 했다.

# Part A. WavLM-base-plus 3모델 (학습: Colab)

## 01. Mission 1 개요

구급 신고 음성의 신고자 성별(M/F)을 분류한다. 평가지표는 Accuracy이며, 최종 제출은 `audio file name,gender` 형식의 CSV를 생성한다.

WavLM-base-plus 기반 세 분류기의 확률 가중 앙상블이다(최종 모델에서 25% 비중).

- Attention+F0: 40%
- Statistics Pooling+F0: 40%
- Statistics Pooling (No F0): 20%

> 이 노트북은 원본 `Mission1 (2).ipynb`의 실제 학습 코드와 실행 로그를 정리한 제출용 사본이다. 이미 완료된 모델 학습이나 전체 Validation 재추론을 다시 수행하지 않는다.

## 02. 데이터 구성 및 전처리

- 학습: 서울 Training 29,200개
- 공식 Validation: 3,640개
- JSON에서 전처리에 사용하는 필드는 신고자 구간을 위한 `startAt`, `endAt`, `Speaker`뿐이다.
- 음성은 신고자 구간만 이어 붙여 16 kHz mono로 처리하고, 최대 8초 단위로 front/middle/end 최대 3 chunks를 만든다.
- 클래스 매핑은 `0=M`, `1=F`다. `gender`는 학습 target과 성능 평가에만 사용한다.

### 초기 기준선 요약

CNN은 Accuracy 93.96%, MFCC+F0 RandomForest는 94.23%였다. Frozen WavLM 1-chunk는 95.66%, 3-chunk 추론은 97.50%까지 향상됐다. 아래부터는 최종 세 모델에 필요한 WavLM/F0/statistics pooling 학습 과정만 보존한다.

## 03. 특징 추출

WavLM mean embedding은 chunk당 768차원이다. Statistics pooling은 WavLM 마지막 hidden state의 frame-level mean 768차원과 std 768차원을 이어 1,536차원으로 구성한다. F0 특징은 6차원이다.

패딩 chunk는 mask로 attention에서 제외하며 scaler는 유효 chunk에만 적용한다.

In [ ]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. 경로
# ============================================================

CACHE_DIR = Path(
    "/content/drive/MyDrive/mission1/"
    "wavlm_embedding_cache/"
    "train_3chunk_separate"
)

shard_paths = sorted(
    CACHE_DIR.glob("shard_*.npz")
)

print(
    "Shard count:",
    len(shard_paths)
)

assert len(shard_paths) == 59


# ============================================================
# 2. WavLM chunk cache 다시 로드
# ============================================================

chunk_list = []
label_list = []
num_chunk_list = []
filename_list = []
index_list = []


for path in shard_paths:

    data = np.load(
        path,
        allow_pickle=True
    )

    chunk_list.append(
        data["chunk_embeddings"]
    )

    label_list.append(
        data["true_idx"]
    )

    num_chunk_list.append(
        data["num_chunks"]
    )

    filename_list.append(
        data["filenames"]
    )

    index_list.append(
        data["sample_indices"]
    )


X_chunks = np.concatenate(
    chunk_list,
    axis=0
).astype(np.float32)

y_wavlm = np.concatenate(
    label_list
).astype(np.int64)

num_chunks = np.concatenate(
    num_chunk_list
).astype(np.int64)

wavlm_filenames = np.concatenate(
    filename_list
)

sample_indices = np.concatenate(
    index_list
).astype(np.int64)


print("\n" + "=" * 70)
print("RAW WAVLM CACHE")
print("=" * 70)

print(
    "X_chunks      :",
    X_chunks.shape
)

print(
    "labels        :",
    y_wavlm.shape
)

print(
    "num_chunks    :",
    num_chunks.shape
)


# ============================================================
# 3. F0와 다시 정렬 확인
# ============================================================

assert np.array_equal(
    y_wavlm,
    f0_y
)

assert np.array_equal(
    sample_indices,
    f0_indices
)

assert np.array_equal(
    wavlm_filenames.astype(str),
    f0_filenames.astype(str)
)

print(
    "F0 alignment  : ✅"
)


# ============================================================
# 4. chunk mask 생성
#
# 1 chunk:
# [1,0,0]
#
# 3 chunks:
# [1,1,1]
# ============================================================

chunk_mask = np.zeros(
    (len(X_chunks), 3),
    dtype=np.float32
)

chunk_mask[:, 0] = 1.0

three_chunk = (
    num_chunks == 3
)

chunk_mask[
    three_chunk,
    1:
] = 1.0


print(
    "1-chunk samples:",
    int((num_chunks == 1).sum())
)

print(
    "3-chunk samples:",
    int((num_chunks == 3).sum())
)


# ============================================================
# 5. 고정 internal split 다시 확인
# ============================================================

split = np.load(
    CACHE_DIR
    / "internal_split_seed42.npz"
)

train_idx = split["train_idx"]
val_idx = split["val_idx"]

y_train = y_wavlm[
    train_idx
]

y_val = y_wavlm[
    val_idx
]


# ============================================================
# 6. train / val 분리
# ============================================================

X_train = X_chunks[
    train_idx
].copy()

X_val = X_chunks[
    val_idx
].copy()

mask_train = chunk_mask[
    train_idx
].copy()

mask_val = chunk_mask[
    val_idx
].copy()


# ============================================================
# 7. StandardScaler
#
# 중요:
# NaN인 missing chunk는 scaler fit에 포함하지 않음.
#
# 예전 Attention 실험과 동일한 방식.
# ============================================================

valid_train_chunks = X_train[
    mask_train.astype(bool)
]

print(
    "\nActual training chunks:",
    valid_train_chunks.shape
)

assert not np.isnan(
    valid_train_chunks
).any()


wavlm_scaler = StandardScaler()

wavlm_scaler.fit(
    valid_train_chunks
)


# ============================================================
# 8. 모든 실제 chunk scaling
# ============================================================

X_train_scaled = np.zeros_like(
    X_train,
    dtype=np.float32
)

X_val_scaled = np.zeros_like(
    X_val,
    dtype=np.float32
)


train_valid = (
    mask_train.astype(bool)
)

val_valid = (
    mask_val.astype(bool)
)


X_train_scaled[
    train_valid
] = wavlm_scaler.transform(
    X_train[
        train_valid
    ]
).astype(np.float32)


X_val_scaled[
    val_valid
] = wavlm_scaler.transform(
    X_val[
        val_valid
    ]
).astype(np.float32)


# ============================================================
# 9. F0도 동일 split으로 준비
#
# F0 scaler 역시 TRAIN에만 fit.
# ============================================================

X_train_f0 = X_f0[
    train_idx
].astype(np.float32)

X_val_f0 = X_f0[
    val_idx
].astype(np.float32)


f0_scaler = StandardScaler()

X_train_f0_scaled = (
    f0_scaler.fit_transform(
        X_train_f0
    )
    .astype(np.float32)
)

X_val_f0_scaled = (
    f0_scaler.transform(
        X_val_f0
    )
    .astype(np.float32)
)


# ============================================================
# 10. 최종 검증
# ============================================================

print("\n" + "=" * 70)
print("RESTORED FUSION DATA")
print("=" * 70)

print(
    "X_train_scaled    :",
    X_train_scaled.shape
)

print(
    "X_val_scaled      :",
    X_val_scaled.shape
)

print(
    "mask_train        :",
    mask_train.shape
)

print(
    "mask_val          :",
    mask_val.shape
)

print(
    "X_train_f0_scaled :",
    X_train_f0_scaled.shape
)

print(
    "X_val_f0_scaled   :",
    X_val_f0_scaled.shape
)

print(
    "y_train           :",
    y_train.shape
)

print(
    "y_val             :",
    y_val.shape
)


# ============================================================
# NaN / Inf
# ============================================================

assert not np.isnan(
    X_train_scaled
).any()

assert not np.isnan(
    X_val_scaled
).any()

assert not np.isnan(
    X_train_f0_scaled
).any()

assert not np.isnan(
    X_val_f0_scaled
).any()

assert not np.isinf(
    X_train_scaled
).any()

assert not np.isinf(
    X_val_scaled
).any()


# missing chunk가 정확히 0인지
missing_train = (
    mask_train == 0
)

missing_val = (
    mask_val == 0
)

if missing_train.any():

    assert np.max(
        np.abs(
            X_train_scaled[
                missing_train
            ]
        )
    ) == 0

if missing_val.any():

    assert np.max(
        np.abs(
            X_val_scaled[
                missing_val
            ]
        )
    ) == 0


print(
    "\nMissing chunk → 0: ✅"
)

print(
    "NaN / Inf       : ✅"
)

print(
    "F0 alignment    : ✅"
)

print(
    "\n🎉 Attention + F0 fusion 데이터 준비 완료"
)

Shard count: 59

RAW WAVLM CACHE
X_chunks      : (29200, 3, 768)
labels        : (29200,)
num_chunks    : (29200,)
F0 alignment  : ✅
1-chunk samples: 140
3-chunk samples: 29060

Actual training chunks: (78576, 768)

RESTORED FUSION DATA
X_train_scaled    : (26280, 3, 768)
X_val_scaled      : (2920, 3, 768)
mask_train        : (26280, 3)
mask_val          : (2920, 3)
X_train_f0_scaled : (26280, 6)
X_val_f0_scaled   : (2920, 6)
y_train           : (26280,)
y_val             : (2920,)

Missing chunk → 0: ✅
NaN / Inf       : ✅
F0 alignment    : ✅

🎉 Attention + F0 fusion 데이터 준비 완료


## 04. 학습 데이터 분할

최종 학습은 서울 Training만 사용하며, 내부 Train/Holdout은 26,280 / 2,920개다. 각 학습 셀은 두 인덱스가 겹치지 않는지 assertion으로 검사하고 scaler를 Train subset에서만 fit한다.

## 05. 최종 모델 학습

### Attention+F0

768차원 WavLM chunk embedding을 attention pooling한 뒤 F0 6차원과 결합한다. 아래 3-seed 학습 코드는 실제 기록된 epoch 로그와 함께 보존한다.

In [ ]:
import random
import copy
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


# ============================================================
# Model
# ============================================================

class AttentionF0Fusion(nn.Module):

    def __init__(
        self,
        input_dim=768,
        f0_dim=6,
        attention_hidden=128,
        classifier_hidden=256,
        dropout=0.2,
    ):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(input_dim, attention_hidden),
            nn.Tanh(),
            nn.Linear(attention_hidden, 1),
        )

        self.classifier = nn.Sequential(
            nn.Linear(input_dim + f0_dim, classifier_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden, 2),
        )

    def forward(self, x, mask, f0):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(
            mask == 0,
            -1e9
        )

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        fusion = torch.cat(
            [pooled, f0],
            dim=1
        )

        logits = self.classifier(
            fusion
        )

        return logits, weights


# ============================================================
# Dataset
# ============================================================

train_dataset = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(mask_train, dtype=torch.float32),
    torch.tensor(X_train_f0_scaled, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)

val_dataset = TensorDataset(
    torch.tensor(X_val_scaled, dtype=torch.float32),
    torch.tensor(mask_val, dtype=torch.float32),
    torch.tensor(X_val_f0_scaled, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long),
)


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# Seed experiment
# ============================================================

SEEDS = [42, 123, 2026]

MAX_EPOCHS = 30
PATIENCE = 5

results = []
all_seed_probs = []
all_seed_preds = []


for seed in SEEDS:

    print("\n" + "=" * 70)
    print(f"SEED {seed}")
    print("=" * 70)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        generator=generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=512,
        shuffle=False,
    )


    model = AttentionF0Fusion().to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=5e-4,
        weight_decay=1e-4,
    )


    best_f1 = -1
    best_epoch = None
    best_state = None
    patience_count = 0


    # ========================================================
    # Train
    # ========================================================

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        total_loss = 0
        total_correct = 0
        total_count = 0


        for xb, mb, f0b, yb in train_loader:

            xb = xb.to(device)
            mb = mb.to(device)
            f0b = f0b.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits, _ = model(
                xb,
                mb,
                f0b
            )

            loss = criterion(
                logits,
                yb
            )

            loss.backward()
            optimizer.step()

            total_loss += (
                loss.item() * len(yb)
            )

            total_correct += (
                logits.argmax(1) == yb
            ).sum().item()

            total_count += len(yb)


        # ====================================================
        # Validation
        # ====================================================

        model.eval()

        preds = []
        truths = []


        with torch.no_grad():

            for xb, mb, f0b, yb in val_loader:

                xb = xb.to(device)
                mb = mb.to(device)
                f0b = f0b.to(device)

                logits, _ = model(
                    xb,
                    mb,
                    f0b
                )

                pred = logits.argmax(
                    dim=1
                )

                preds.append(
                    pred.cpu().numpy()
                )

                truths.append(
                    yb.numpy()
                )


        pred_epoch = np.concatenate(preds)
        true_epoch = np.concatenate(truths)

        acc = accuracy_score(
            true_epoch,
            pred_epoch
        )

        f1 = f1_score(
            true_epoch,
            pred_epoch,
            average="macro"
        )

        errors = int(
            (pred_epoch != true_epoch).sum()
        )


        print(
            f"Epoch {epoch:02d} | "
            f"loss={total_loss/total_count:.4f} | "
            f"train_acc={total_correct/total_count:.4f} | "
            f"val_acc={acc:.6f} | "
            f"f1={f1:.6f} | "
            f"errors={errors}"
        )


        if f1 > best_f1:

            best_f1 = f1
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_count = 0

        else:

            patience_count += 1


        if patience_count >= PATIENCE:
            break


    # ========================================================
    # Best model inference
    # ========================================================

    model.load_state_dict(
        best_state
    )

    model.eval()

    probs_list = []
    pred_list = []
    true_list = []
    weight_list = []


    with torch.no_grad():

        for xb, mb, f0b, yb in val_loader:

            xb = xb.to(device)
            mb = mb.to(device)
            f0b = f0b.to(device)

            logits, weights = model(
                xb,
                mb,
                f0b
            )

            probs = torch.softmax(
                logits,
                dim=1
            )

            probs_list.append(
                probs.cpu().numpy()
            )

            pred_list.append(
                probs.argmax(1).cpu().numpy()
            )

            true_list.append(
                yb.numpy()
            )

            weight_list.append(
                weights.cpu().numpy()
            )


    probs = np.concatenate(
        probs_list
    )

    pred = np.concatenate(
        pred_list
    )

    true = np.concatenate(
        true_list
    )

    weights = np.concatenate(
        weight_list
    )


    acc = accuracy_score(
        true,
        pred
    )

    f1 = f1_score(
        true,
        pred,
        average="macro"
    )

    errors = int(
        (pred != true).sum()
    )


    three_val = (
        mask_val.sum(axis=1) == 3
    )

    avg_weights = weights[
        three_val
    ].mean(axis=0)


    print("\nBEST")

    print(
        f"epoch={best_epoch}"
    )

    print(
        f"accuracy={acc:.6f}"
    )

    print(
        f"macro_f1={f1:.6f}"
    )

    print(
        f"errors={errors}"
    )

    print(
        "weights:",
        avg_weights
    )


    results.append({
        "seed": seed,
        "best_epoch": best_epoch,
        "accuracy": acc,
        "macro_f1": f1,
        "errors": errors,
        "front_weight": avg_weights[0],
        "middle_weight": avg_weights[1],
        "end_weight": avg_weights[2],
    })

    all_seed_probs.append(
        probs
    )

    all_seed_preds.append(
        pred
    )


# ============================================================
# 3-seed summary
# ============================================================

accs = np.array([
    r["accuracy"]
    for r in results
])

f1s = np.array([
    r["macro_f1"]
    for r in results
])

errors = np.array([
    r["errors"]
    for r in results
])


print("\n" + "=" * 70)
print("ATTENTION + F0 — 3 SEED SUMMARY")
print("=" * 70)

for r in results:

    print(
        f"seed {r['seed']:4d} | "
        f"epoch {r['best_epoch']:2d} | "
        f"acc={r['accuracy']:.6f} | "
        f"F1={r['macro_f1']:.6f} | "
        f"errors={r['errors']}"
    )


print()

print(
    f"Mean Accuracy : "
    f"{accs.mean():.6f} "
    f"± {accs.std():.6f}"
)

print(
    f"Mean Macro F1 : "
    f"{f1s.mean():.6f} "
    f"± {f1s.std():.6f}"
)

print(
    f"Mean Errors   : "
    f"{errors.mean():.2f}"
)


# ============================================================
# Probability ensemble
# ============================================================

ensemble_probs = np.mean(
    np.stack(
        all_seed_probs,
        axis=0
    ),
    axis=0
)

ensemble_pred = ensemble_probs.argmax(
    axis=1
)

ensemble_acc = accuracy_score(
    y_val,
    ensemble_pred
)

ensemble_f1 = f1_score(
    y_val,
    ensemble_pred,
    average="macro"
)

ensemble_errors = int(
    (ensemble_pred != y_val).sum()
)


print("\n" + "=" * 70)
print("F0 FUSION — 3-SEED PROBABILITY ENSEMBLE")
print("=" * 70)

print(
    f"Accuracy : {ensemble_acc:.6f}"
)

print(
    f"Macro F1 : {ensemble_f1:.6f}"
)

print(
    f"Errors   : {ensemble_errors}"
)

print(
    "\nConfusion [M,F]:"
)

print(
    confusion_matrix(
        y_val,
        ensemble_pred,
        labels=[0, 1]
    )
)


# ============================================================
# 저장
# ============================================================

save_path = (
    CACHE_DIR
    / "internal_attention_f0_3seed_result.npz"
)

np.savez_compressed(
    save_path,

    true=y_val,

    seed42_probs=all_seed_probs[0],
    seed123_probs=all_seed_probs[1],
    seed2026_probs=all_seed_probs[2],

    seed42_pred=all_seed_preds[0],
    seed123_pred=all_seed_preds[1],
    seed2026_pred=all_seed_preds[2],

    ensemble_probs=ensemble_probs,
    ensemble_pred=ensemble_pred,
)


print(
    "\nSaved:",
    save_path
)

Device: cpu

SEED 42
Epoch 01 | loss=0.1473 | train_acc=0.9530 | val_acc=0.970890 | f1=0.970741 | errors=85
Epoch 02 | loss=0.0809 | train_acc=0.9755 | val_acc=0.975000 | f1=0.974879 | errors=73
Epoch 03 | loss=0.0724 | train_acc=0.9781 | val_acc=0.975342 | f1=0.975229 | errors=72
Epoch 04 | loss=0.0680 | train_acc=0.9802 | val_acc=0.976370 | f1=0.976264 | errors=69
Epoch 05 | loss=0.0624 | train_acc=0.9814 | val_acc=0.976027 | f1=0.975912 | errors=70
Epoch 06 | loss=0.0584 | train_acc=0.9821 | val_acc=0.978082 | f1=0.977973 | errors=64
Epoch 07 | loss=0.0541 | train_acc=0.9844 | val_acc=0.977740 | f1=0.977629 | errors=65
Epoch 08 | loss=0.0506 | train_acc=0.9852 | val_acc=0.975342 | f1=0.975224 | errors=72
Epoch 09 | loss=0.0462 | train_acc=0.9874 | val_acc=0.977397 | f1=0.977293 | errors=66
Epoch 10 | loss=0.0441 | train_acc=0.9879 | val_acc=0.977740 | f1=0.977629 | errors=65
Epoch 11 | loss=0.0412 | train_acc=0.9887 | val_acc=0.976370 | f1=0.976255 | errors=69

BEST
epoch=6
accuracy

### Statistics Pooling cache 검증

각 shard의 filename, label, chunk 수, sample index가 기존 WavLM cache와 같은지 확인한다. 또한 유효 chunk만 scaler에 입력되도록 mask를 만든다.

In [ ]:
import numpy as np
from pathlib import Path

STATS_DIR = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache/train_stats_3chunk"
)
OLD_DIR = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache/train_3chunk_separate"
)

N = 29200

# 학습에 사용할 배열
X_stats = np.full((N, 3, 1536), np.nan, dtype=np.float32)
y_stats = np.empty(N, dtype=np.int64)
n_chunks_stats = np.empty(N, dtype=np.int64)

max_diff = 0.0
sum_diff = 0.0
compared = 0

print("=" * 65)
print("FINAL VERIFICATION + TRAINING DATA PREPARATION")
print("=" * 65)

for start in range(0, N, 500):
    end = min(start + 500, N)

    stats_path = STATS_DIR / f"stats_{start:05d}_{end-1:05d}.npz"
    old_path = OLD_DIR / f"shard_{start:05d}_{end-1:05d}.npz"

    assert stats_path.exists()
    assert old_path.exists()

    with np.load(stats_path, allow_pickle=False) as s, \
         np.load(old_path, allow_pickle=False) as o:

        x = s["chunk_stats"]
        y = s["true_idx"]
        counts = s["num_chunks"]

        assert x.shape == (end - start, 3, 1536)
        assert np.array_equal(
            s["sample_indices"], np.arange(start, end)
        )
        assert np.array_equal(y, o["true_idx"])
        assert np.array_equal(counts, o["num_chunks"])
        assert np.array_equal(
            s["filenames"].astype(str),
            o["filenames"].astype(str)
        )

        valid = np.arange(3)[None, :] < counts[:, None]

        assert np.isfinite(x[valid]).all()
        assert np.isnan(x[~valid]).all()

        diff = np.abs(x[:, :, :768][valid] - o["chunk_embeddings"][valid])

        max_diff = max(max_diff, float(diff.max()))
        sum_diff += float(diff.sum(dtype=np.float64))
        compared += diff.size

        X_stats[start:end] = x
        y_stats[start:end] = y
        n_chunks_stats[start:end] = counts

    if end % 5000 == 0 or end == N:
        print(f"Verified: {end:,} / {N:,}")

assert np.isin(n_chunks_stats, [1, 3]).all()
assert int((n_chunks_stats == 1).sum()) == 140
assert int((n_chunks_stats == 3).sum()) == 29060

# 유효한 chunk만 1로 표시
mask_stats = (
    np.arange(3)[None, :] < n_chunks_stats[:, None]
).astype(np.float32)

# 없는 chunk의 NaN을 0으로 변경
X_stats = np.nan_to_num(X_stats, nan=0.0)

print("\n" + "=" * 65)
print("FINAL RESULTS")
print("=" * 65)
print("X_stats shape :", X_stats.shape)
print("y_stats shape :", y_stats.shape)
print("mask shape    :", mask_stats.shape)
print("Max mean diff :", max_diff)
print("Avg mean diff :", sum_diff / compared)

assert max_diff < 1e-3
assert np.isfinite(X_stats).all()

print("\n✅ ALL CHECKS PASSED")
print("✅ TRAINING DATA READY")

FINAL VERIFICATION + TRAINING DATA PREPARATION
Verified: 5,000 / 29,200
Verified: 10,000 / 29,200
Verified: 15,000 / 29,200
Verified: 20,000 / 29,200
Verified: 25,000 / 29,200
Verified: 29,200 / 29,200

FINAL RESULTS
X_stats shape : (29200, 3, 1536)
y_stats shape : (29200,)
mask shape    : (29200, 3)
Max mean diff : 3.316998481750488e-05
Avg mean diff : 3.6059952915874304e-08

✅ ALL CHECKS PASSED
✅ TRAINING DATA READY


### Stats Attention (No F0)

1,536차원 statistics feature에 attention pooling과 MLP classifier를 적용한다.

In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# 1. 설정
# ============================================================

SEED = 42
EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 256

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SPLIT_PATH = Path(
    "/content/drive/MyDrive/mission1/"
    "wavlm_embedding_cache/train_3chunk_separate/"
    "internal_split_seed42.npz"
)

with np.load(SPLIT_PATH) as split:
    train_idx = split["train_idx"]
    val_idx = split["val_idx"]

assert len(train_idx) == 26280
assert len(val_idx) == 2920
assert len(np.intersect1d(train_idx, val_idx)) == 0

# ============================================================
# 2. Feature Scaling
# ============================================================

# 유효한 chunk만 scaler 학습에 사용
train_mask = mask_stats[train_idx].astype(bool)

scaler = StandardScaler()

scaler.fit(
    X_stats[train_idx][train_mask]
)

# 29,200개 전체를 복사하지 않고 필요한 부분만 변환
def prepare_features(indices):
    x = X_stats[indices].copy()
    mask = mask_stats[indices].astype(bool)

    x[mask] = scaler.transform(x[mask])
    x[~mask] = 0.0

    return x.astype(np.float32)

X_train = prepare_features(train_idx)
X_val = prepare_features(val_idx)

y_train = y_stats[train_idx]
y_val = y_stats[val_idx]

m_train = mask_stats[train_idx]
m_val = mask_stats[val_idx]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

# ============================================================
# 3. DataLoader
# ============================================================

train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(m_train),
    torch.from_numpy(y_train)
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(m_val),
    torch.from_numpy(y_val)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ============================================================
# 4. Statistics Attention Model
# ============================================================

class StatisticsAttentionMLP(nn.Module):

    def __init__(
        self,
        input_dim=1536,
        attention_hidden=128,
        classifier_hidden=256,
        dropout=0.2
    ):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(input_dim, attention_hidden),
            nn.Tanh(),
            nn.Linear(attention_hidden, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, classifier_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden, 2)
        )

    def forward(self, x, mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(
            mask == 0,
            -1e9
        )

        weights = torch.softmax(scores, dim=1)

        pooled = (
            x * weights.unsqueeze(-1)
        ).sum(dim=1)

        logits = self.classifier(pooled)

        return logits, weights

model = StatisticsAttentionMLP().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()

# ============================================================
# 5. Training
# ============================================================

best_f1 = -1
best_epoch = 0
best_state = None
patience_count = 0

print("\n" + "=" * 65)
print("STATISTICS POOLING + ATTENTION TRAINING")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):

    model.train()
    total_loss = 0.0

    for xb, mb, yb in train_loader:

        xb = xb.to(DEVICE)
        mb = mb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        logits, _ = model(xb, mb)

        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)

    model.eval()

    all_probs = []
    all_labels = []

    with torch.inference_mode():

        for xb, mb, yb in val_loader:

            xb = xb.to(DEVICE)
            mb = mb.to(DEVICE)

            logits, _ = model(xb, mb)

            probs = torch.softmax(
                logits, dim=1
            )

            all_probs.append(probs.cpu().numpy())
            all_labels.append(yb.numpy())

    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)

    preds = probs.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")

    print(
        f"Epoch {epoch:02d} | "
        f"Loss {total_loss / len(train_dataset):.4f} | "
        f"Acc {acc:.6f} | "
        f"Macro F1 {f1:.6f}"
    )

    if f1 > best_f1:

        best_f1 = f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_count = 0

    else:

        patience_count += 1

    if patience_count >= PATIENCE:
        print("Early stopping")
        break

# ============================================================
# 6. Best Model Evaluation
# ============================================================

model.load_state_dict(best_state)
model.eval()

all_probs = []

with torch.inference_mode():

    for xb, mb, _ in val_loader:

        xb = xb.to(DEVICE)
        mb = mb.to(DEVICE)

        logits, _ = model(xb, mb)

        all_probs.append(
            torch.softmax(logits, dim=1).cpu().numpy()
        )

best_probs = np.concatenate(all_probs)
best_preds = best_probs.argmax(axis=1)

best_acc = accuracy_score(y_val, best_preds)
best_macro_f1 = f1_score(
    y_val, best_preds, average="macro"
)

print("\n" + "=" * 65)
print("BEST STATISTICS ATTENTION RESULT")
print("=" * 65)

print("Best epoch :", best_epoch)
print("Accuracy   :", best_acc)
print("Macro F1   :", best_macro_f1)
print("Errors     :", int((best_preds != y_val).sum()))
print("Confusion matrix:")
print(confusion_matrix(y_val, best_preds))

# ============================================================
# 7. 결과 저장
# ============================================================

SAVE_DIR = Path(
    "/content/drive/MyDrive/mission1/"
    "wavlm_embedding_cache/train_stats_3chunk"
)

torch.save(
    {
        "model_state_dict": best_state,
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
        "best_epoch": best_epoch,
        "best_macro_f1": best_macro_f1,
        "seed": SEED,
        "input_dim": 1536
    },
    SAVE_DIR / "internal_stats_attention_seed42.pt"
)

np.savez_compressed(
    SAVE_DIR / "internal_stats_attention_seed42_result.npz",
    val_idx=val_idx,
    y_true=y_val,
    probs=best_probs,
    preds=best_preds
)

print("\n✅ Model and predictions saved")

Training: (26280, 3, 1536)
Validation: (2920, 3, 1536)

STATISTICS POOLING + ATTENTION TRAINING
Epoch 01 | Loss 0.1434 | Acc 0.973973 | Macro F1 0.973860
Epoch 02 | Loss 0.0774 | Acc 0.972945 | Macro F1 0.972841
Epoch 03 | Loss 0.0676 | Acc 0.975342 | Macro F1 0.975229
Epoch 04 | Loss 0.0579 | Acc 0.973630 | Macro F1 0.973510
Epoch 05 | Loss 0.0517 | Acc 0.975685 | Macro F1 0.975583
Epoch 06 | Loss 0.0477 | Acc 0.970890 | Macro F1 0.970752
Epoch 07 | Loss 0.0398 | Acc 0.974658 | Macro F1 0.974538
Epoch 08 | Loss 0.0330 | Acc 0.972260 | Macro F1 0.972163
Epoch 09 | Loss 0.0313 | Acc 0.974315 | Macro F1 0.974190
Epoch 10 | Loss 0.0248 | Acc 0.973973 | Macro F1 0.973855
Early stopping

BEST STATISTICS ATTENTION RESULT
Best epoch : 5
Accuracy   : 0.9756849315068493
Macro F1   : 0.9755826359705744
Errors     : 71
Confusion matrix:
[[1330   36]
 [  35 1519]]

✅ Model and predictions saved


### Stats+F0 Attention

1,536차원 statistics feature에 F0 6차원을 결합한다. F0 shard와 stats cache의 순서는 assertion으로 확인한다.

In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# 1. 설정
# ============================================================

SEED = 42
EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 256

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache"
)

STATS_DIR = BASE / "train_stats_3chunk"
F0_DIR = BASE / "train_f0_6d"

with np.load(
    BASE / "train_3chunk_separate/internal_split_seed42.npz"
) as split:
    train_idx = split["train_idx"]
    val_idx = split["val_idx"]

assert len(train_idx) == 26280
assert len(val_idx) == 2920
assert len(np.intersect1d(train_idx, val_idx)) == 0

# ============================================================
# 2. F0 feature 불러오기 + 순서 검증
# ============================================================

X_f0 = np.empty((29200, 6), dtype=np.float32)

f0_files = sorted(F0_DIR.glob("f0_*.npz"))
assert len(f0_files) == 59, f"F0 shards: {len(f0_files)}"

for start in range(0, 29200, 500):
    end = min(start + 500, 29200)

    path = F0_DIR / f"f0_{start:05d}_{end-1:05d}.npz"

    with np.load(path, allow_pickle=False) as data:
        assert np.array_equal(
            data["sample_indices"],
            np.arange(start, end)
        )
        assert np.array_equal(
            data["true_idx"],
            y_stats[start:end]
        )
        assert np.array_equal(
            data["filenames"].astype(str),
            wavlm_filenames[start:end].astype(str)
        )

        X_f0[start:end] = data["f0_features"]

assert np.isfinite(X_f0).all()

print("F0 loaded:", X_f0.shape)

# ============================================================
# 3. Scaler — Training 데이터만 사용
# ============================================================

stats_scaler = StandardScaler()
f0_scaler = StandardScaler()

train_valid_mask = mask_stats[train_idx].astype(bool)

stats_scaler.fit(
    X_stats[train_idx][train_valid_mask]
)

f0_scaler.fit(
    X_f0[train_idx]
)

def prepare_stats(indices):
    x = X_stats[indices].copy()
    mask = mask_stats[indices].astype(bool)

    x[mask] = stats_scaler.transform(x[mask])
    x[~mask] = 0.0

    return x.astype(np.float32)

X_train_stats = prepare_stats(train_idx)
X_val_stats = prepare_stats(val_idx)

X_train_f0 = f0_scaler.transform(
    X_f0[train_idx]
).astype(np.float32)

X_val_f0 = f0_scaler.transform(
    X_f0[val_idx]
).astype(np.float32)

y_train = y_stats[train_idx]
y_val = y_stats[val_idx]

m_train = mask_stats[train_idx]
m_val = mask_stats[val_idx]

print("Train stats :", X_train_stats.shape)
print("Train F0    :", X_train_f0.shape)
print("Val stats   :", X_val_stats.shape)
print("Val F0      :", X_val_f0.shape)

# ============================================================
# 4. DataLoader
# ============================================================

train_dataset = TensorDataset(
    torch.from_numpy(X_train_stats),
    torch.from_numpy(X_train_f0),
    torch.from_numpy(m_train),
    torch.from_numpy(y_train)
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val_stats),
    torch.from_numpy(X_val_f0),
    torch.from_numpy(m_val),
    torch.from_numpy(y_val)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ============================================================
# 5. Statistics + F0 Attention 모델
# ============================================================

class StatisticsF0AttentionMLP(nn.Module):

    def __init__(
        self,
        input_dim=1536,
        f0_dim=6,
        attention_hidden=128,
        classifier_hidden=256,
        dropout=0.2
    ):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(input_dim, attention_hidden),
            nn.Tanh(),
            nn.Linear(attention_hidden, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                input_dim + f0_dim,
                classifier_hidden
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden, 2)
        )

    def forward(self, x, f0, mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(
            mask == 0,
            -1e9
        )

        weights = torch.softmax(scores, dim=1)

        pooled = (
            x * weights.unsqueeze(-1)
        ).sum(dim=1)

        fused = torch.cat(
            [pooled, f0],
            dim=1
        )

        logits = self.classifier(fused)

        return logits, weights

model = StatisticsF0AttentionMLP().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()

# ============================================================
# 6. 학습
# ============================================================

best_f1 = -1
best_epoch = 0
best_state = None
patience_count = 0

print("\n" + "=" * 65)
print("STATISTICS + F0 ATTENTION TRAINING")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):

    model.train()
    total_loss = 0.0

    for xb, fb, mb, yb in train_loader:

        xb = xb.to(DEVICE)
        fb = fb.to(DEVICE)
        mb = mb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        logits, _ = model(xb, fb, mb)

        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)

    model.eval()

    all_probs = []
    all_labels = []

    with torch.inference_mode():

        for xb, fb, mb, yb in val_loader:

            xb = xb.to(DEVICE)
            fb = fb.to(DEVICE)
            mb = mb.to(DEVICE)

            logits, _ = model(xb, fb, mb)

            probs = torch.softmax(
                logits, dim=1
            )

            all_probs.append(probs.cpu().numpy())
            all_labels.append(yb.numpy())

    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)

    preds = probs.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")

    print(
        f"Epoch {epoch:02d} | "
        f"Loss {total_loss / len(train_dataset):.4f} | "
        f"Acc {acc:.6f} | "
        f"Macro F1 {f1:.6f}"
    )

    if f1 > best_f1:
        best_f1 = f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_count = 0
    else:
        patience_count += 1

    if patience_count >= PATIENCE:
        print("Early stopping")
        break

# ============================================================
# 7. Best 모델 평가
# ============================================================

model.load_state_dict(best_state)
model.eval()

all_probs = []

with torch.inference_mode():

    for xb, fb, mb, _ in val_loader:

        xb = xb.to(DEVICE)
        fb = fb.to(DEVICE)
        mb = mb.to(DEVICE)

        logits, _ = model(xb, fb, mb)

        all_probs.append(
            torch.softmax(logits, dim=1).cpu().numpy()
        )

best_probs = np.concatenate(all_probs)
best_preds = best_probs.argmax(axis=1)

best_acc = accuracy_score(y_val, best_preds)
best_macro_f1 = f1_score(
    y_val, best_preds, average="macro"
)

print("\n" + "=" * 65)
print("BEST STATISTICS + F0 RESULT")
print("=" * 65)

print("Best epoch :", best_epoch)
print("Accuracy   :", best_acc)
print("Macro F1   :", best_macro_f1)
print("Errors     :", int((best_preds != y_val).sum()))
print("Confusion matrix:")
print(confusion_matrix(y_val, best_preds))

# ============================================================
# 8. 결과 저장
# ============================================================

torch.save(
    {
        "model_state_dict": best_state,
        "stats_scaler_mean": stats_scaler.mean_,
        "stats_scaler_scale": stats_scaler.scale_,
        "f0_scaler_mean": f0_scaler.mean_,
        "f0_scaler_scale": f0_scaler.scale_,
        "best_epoch": best_epoch,
        "best_macro_f1": best_macro_f1,
        "seed": SEED,
        "input_dim": 1536,
        "f0_dim": 6
    },
    STATS_DIR / "internal_stats_f0_attention_seed42.pt"
)

np.savez_compressed(
    STATS_DIR / "internal_stats_f0_attention_seed42_result.npz",
    val_idx=val_idx,
    y_true=y_val,
    probs=best_probs,
    preds=best_preds
)

print("\n✅ Model and predictions saved")

F0 loaded: (29200, 6)
Train stats : (26280, 3, 1536)
Train F0    : (26280, 6)
Val stats   : (2920, 3, 1536)
Val F0      : (2920, 6)

STATISTICS + F0 ATTENTION TRAINING
Epoch 01 | Loss 0.1465 | Acc 0.967808 | Macro F1 0.967618
Epoch 02 | Loss 0.0759 | Acc 0.975000 | Macro F1 0.974871
Epoch 03 | Loss 0.0647 | Acc 0.976370 | Macro F1 0.976268
Epoch 04 | Loss 0.0562 | Acc 0.973973 | Macro F1 0.973852
Epoch 05 | Loss 0.0504 | Acc 0.977740 | Macro F1 0.977636
Epoch 06 | Loss 0.0431 | Acc 0.977055 | Macro F1 0.976952
Epoch 07 | Loss 0.0411 | Acc 0.974315 | Macro F1 0.974183
Epoch 08 | Loss 0.0319 | Acc 0.973288 | Macro F1 0.973167
Epoch 09 | Loss 0.0315 | Acc 0.974658 | Macro F1 0.974523
Epoch 10 | Loss 0.0252 | Acc 0.966438 | Macro F1 0.966267
Early stopping

BEST STATISTICS + F0 RESULT
Best epoch : 5
Accuracy   : 0.9777397260273972
Macro F1   : 0.9776358554031974
Errors     : 65
Confusion matrix:
[[1328   38]
 [  27 1527]]

✅ Model and predictions saved


## 06. 앙상블 및 모델 선택

아래 비교는 동일한 내부 holdout prediction을 사용한다. 최종 가중치 40:40:20은 Attention+F0, Stats+F0, Stats(No F0)의 확률을 각각 0.40, 0.40, 0.20으로 결합한다.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    "/content/gdrive_new/MyDrive/mission1/wavlm_embedding_cache"
)

ATT = ROOT / "train_attention_f0"
STATS = ROOT / "train_stats_3chunk"

# ============================================================
# 1. 세 모델의 Holdout 예측 불러오기
# ============================================================

files = {
    "attention_f0": (
        ATT / "internal_attention_f0_seed_42_result.npz"
    ),
    "stats_f0": (
        STATS / "internal_stats_f0_attention_seed42_result.npz"
    ),
    "stats": (
        STATS / "internal_stats_attention_seed42_result.npz"
    ),
}

data = {}

for name, path in files.items():
    assert path.is_file(), f"파일 없음: {path}"

    with np.load(path) as d:
        data[name] = {
            "idx": d["val_idx"].copy(),
            "y": d["y_true"].astype(int).copy(),
            "probs": d["probs"].copy(),
        }

base = data["attention_f0"]

for name, item in data.items():
    assert np.array_equal(
        base["idx"], item["idx"]
    ), f"{name}: 인덱스 불일치"

    assert np.array_equal(
        base["y"], item["y"]
    ), f"{name}: 정답 불일치"

y = base["y"]

a = data["attention_f0"]["probs"]
b = data["stats_f0"]["probs"]
c = data["stats"]["probs"]

# ============================================================
# 2. 앙상블 조합 실험
# ============================================================

candidates = {
    "Attention F0": a,
    "Stats F0": b,
    "Stats": c,
    "Current 50:50": 0.5*a + 0.5*b,
    "Three models equal": (a+b+c)/3,
    "45:45:10": 0.45*a + 0.45*b + 0.10*c,
    "40:40:20": 0.40*a + 0.40*b + 0.20*c,
    "35:35:30": 0.35*a + 0.35*b + 0.30*c,
    "30:30:40": 0.30*a + 0.30*b + 0.40*c,
    "25:25:50": 0.25*a + 0.25*b + 0.50*c,
}

rows = []

for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    errors = int((pred != y).sum())

    rows.append({
        "ensemble": name,
        "errors": errors,
        "accuracy": 100 * (1 - errors/len(y)),
    })

results = pd.DataFrame(rows).sort_values(
    ["errors", "ensemble"]
)

print("=" * 65)
print("STEP 36: THIRD MODEL CONTRIBUTION")
print("=" * 65)

print("\n[1] 모델 조합별 Holdout 성능")
print(
    results.to_string(
        index=False,
        formatters={"accuracy": "{:.4f}".format}
    )
)

# ============================================================
# 3. 현재 앙상블과 세 번째 모델의 오류 관계
# ============================================================

current_pred = (0.5*a + 0.5*b).argmax(axis=1)
stats_pred = c.argmax(axis=1)

current_wrong = current_pred != y
stats_wrong = stats_pred != y

print("\n[2] 오류 겹침")
print("현재 앙상블 오답:", int(current_wrong.sum()))
print("Stats 모델 오답:", int(stats_wrong.sum()))

print(
    "현재 앙상블 오답 / Stats 정답:",
    int((current_wrong & ~stats_wrong).sum())
)

print(
    "현재 앙상블 정답 / Stats 오답:",
    int((~current_wrong & stats_wrong).sum())
)

print(
    "둘 다 오답:",
    int((current_wrong & stats_wrong).sum())
)

# ============================================================
# 4. 세 모델 모두 틀린 샘플
# ============================================================

wrong_a = a.argmax(axis=1) != y
wrong_b = b.argmax(axis=1) != y
wrong_c = c.argmax(axis=1) != y

all_wrong = wrong_a & wrong_b & wrong_c

print("\n[3] 세 모델 오류 분석")
print("세 모델 모두 오답:", int(all_wrong.sum()))

print(
    "세 모델 중 하나라도 정답:",
    int((~all_wrong).sum())
)

print("\n[4] 개선 가능성")
current_errors = int(current_wrong.sum())

best_errors = int(results["errors"].min())

print("현재 오답:", current_errors)
print("실험 조합 최소 오답:", best_errors)
print("개선 개수:", current_errors - best_errors)

print("\n✅ STEP 36 COMPLETE")

STEP 36: THIRD MODEL CONTRIBUTION

[1] 모델 조합별 Holdout 성능
          ensemble  errors accuracy
          40:40:20      61  97.9110
          45:45:10      62  97.8767
     Current 50:50      63  97.8425
          35:35:30      64  97.8082
          Stats F0      65  97.7740
Three models equal      65  97.7740
          25:25:50      66  97.7397
      Attention F0      66  97.7397
          30:30:40      68  97.6712
             Stats      71  97.5685

[2] 오류 겹침
현재 앙상블 오답: 63
Stats 모델 오답: 71
현재 앙상블 오답 / Stats 정답: 12
현재 앙상블 정답 / Stats 오답: 20
둘 다 오답: 51

[3] 세 모델 오류 분석
세 모델 모두 오답: 45
세 모델 중 하나라도 정답: 2875

[4] 개선 가능성
현재 오답: 63
실험 조합 최소 오답: 61
개선 개수: 2

✅ STEP 36 COMPLETE


## 07. 성능 평가

다음 평가는 저장된 공식 Validation 확률을 사용한다. **전체 Validation WAV를 최종 스크립트로 다시 추론한 결과가 아니다.** 최종 3모델 앙상블의 Accuracy는 97.8846% (3,563/3,640), 오류는 77개다.

> 최종 제출 폴더로 Validation WAV 전체를 다시 추론했을 때도 이 3모델 앙상블의 확률은 저장값과 최대 3e-5 차이로 같았고 오류도 77개였다(Part B 6절).

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    "/content/gdrive_new/MyDrive/mission1/wavlm_embedding_cache"
)

# ============================================================
# 1. 공식 Validation 예측 불러오기
# ============================================================

with np.load(
    ROOT / "train_stats_3chunk/official_validation_stats_results.npz"
) as d:
    filenames = d["filenames"].copy()
    y = d["y_true"].astype(int).copy()

    attention_probs = d["attention_probs"].copy()
    stats_probs = d["stats_probs"].copy()
    stats_f0_probs = d["stats_f0_probs"].copy()

with np.load(
    ROOT / "train_attention_f0/official_validation_f0_comparison.npz"
) as d:
    f0_filenames = d["filenames"].copy()
    f0_y = d["y_true"].astype(int).copy()
    attention_f0_probs = d["new_f0_probs"].copy()

assert np.array_equal(filenames, f0_filenames)
assert np.array_equal(y, f0_y)

# ============================================================
# 2. 기존 모델 vs 새로운 후보
# ============================================================

old_probs = (
    0.50 * attention_f0_probs
    + 0.50 * stats_f0_probs
)

new_probs = (
    0.40 * attention_f0_probs
    + 0.40 * stats_f0_probs
    + 0.20 * stats_probs
)

old_pred = old_probs.argmax(axis=1)
new_pred = new_probs.argmax(axis=1)

old_wrong = old_pred != y
new_wrong = new_pred != y

old_errors = int(old_wrong.sum())
new_errors = int(new_wrong.sum())

rescued = old_wrong & ~new_wrong
broken = ~old_wrong & new_wrong

# ============================================================
# 3. 결과 출력
# ============================================================

print("=" * 65)
print("STEP 37: OFFICIAL VALIDATION 3-MODEL ENSEMBLE")
print("=" * 65)

print("\n[1] 정확도 비교")
print(
    "기존 50:50:",
    f"{100 * (1-old_errors/len(y)):.4f}%",
    f"| 오답 {old_errors}개"
)

print(
    "새로운 40:40:20:",
    f"{100 * (1-new_errors/len(y)):.4f}%",
    f"| 오답 {new_errors}개"
)

print("\n[2] 예측 변화")
print("기존 오답 → 새로운 정답:", int(rescued.sum()))
print("기존 정답 → 새로운 오답:", int(broken.sum()))
print("총 오답 감소:", old_errors - new_errors)

print("\n[3] 목표 달성 여부")
print("98% 달성:", new_errors <= 72)
print("목표까지 추가로 줄여야 할 오답:", max(0, new_errors - 72))

# ============================================================
# 4. 변경된 샘플 확인
# ============================================================

gender_map = {0: "M", 1: "F"}

changed = old_pred != new_pred

changed_df = pd.DataFrame({
    "filename": filenames[changed],
    "true_gender": [
        gender_map[int(v)] for v in y[changed]
    ],
    "old_prediction": [
        gender_map[int(v)] for v in old_pred[changed]
    ],
    "new_prediction": [
        gender_map[int(v)] for v in new_pred[changed]
    ],
    "old_confidence": old_probs.max(axis=1)[changed],
    "new_confidence": new_probs.max(axis=1)[changed],
})

print("\n[4] 예측이 변경된 샘플")
print("변경 개수:", len(changed_df))

if len(changed_df):
    print(changed_df.to_string(index=False))

print("\n✅ STEP 37 COMPLETE")

STEP 37: OFFICIAL VALIDATION 3-MODEL ENSEMBLE

[1] 정확도 비교
기존 50:50: 97.8571% | 오답 78개
새로운 40:40:20: 97.8846% | 오답 77개

[2] 예측 변화
기존 오답 → 새로운 정답: 6
기존 정답 → 새로운 오답: 5
총 오답 감소: 1

[3] 목표 달성 여부
98% 달성: False
목표까지 추가로 줄여야 할 오답: 5

[4] 예측이 변경된 샘플
변경 개수: 11
                             filename true_gender old_prediction new_prediction  old_confidence  new_confidence
651e496f5d60e22224166e42_20220106.wav           F              F              M        0.544609        0.543322
651e4aaacb7c8f4a6a1ff79d_20220501.wav           M              M              F        0.531793        0.546228
651e4dfb2795332cde28edfa_20220703.wav           F              M              F        0.524616        0.564762
651e4f3cbd89bfb121b468a0_20220807.wav           F              M              F        0.559523        0.501968
651e4fb51952d0ca070e692f_20220906.wav           M              F              M        0.513320        0.538824
651e5029d029fc04ba3a722d_20221003.wav           M              F             

## 09. 한계 및 향후 개선

- 77개 오류 중 세 모델이 공통으로 틀리는 고확신도 샘플은 현 특징만으로 구분이 어렵다.
- F0는 WavLM 표현을 보완하지만 단독 모델 성능을 보장하지 않는다.
- WavLM feature 추출과 local pretrained model 포함으로 추론 비용과 제출 용량이 크다.
- 이후 가중치와 threshold 탐색은 공식 Validation이 아니라 내부 holdout 또는 OOF prediction에서 수행한다.

→ 첫 번째 한계(세 모델이 함께 틀리는 샘플)를 줄이기 위해 Part B에서 다른 정보원(ECAPA)을 더한다.

# Part B. ECAPA 화자 임베딩 성별 모델 (학습: 로컬 CPU)

Part A의 3모델은 모두 WavLM 입력이라 오류가 많이 겹친다(앙상블 오류 77개 중 52개를 세 모델이 모두 틀림).
그래서 **다른 방식으로 학습된 화자 인식 임베딩(ECAPA-TDNN)**으로 성별 모델을 따로 만들고, 내부 holdout으로 고른 비율로 섞는다.

**규칙 준수**
- 학습 데이터: 서울 Training 29,200통화만 사용. Validation 3,640통화는 학습(역전파·적합)에 쓰지 않고 마지막 확인에만 사용.
- annotation: 입력에는 `startAt`, `endAt`, `speaker`만 사용(신고자 구간 선택). `gender`는 학습 정답과 채점에만 사용.
- 사전학습 모델: `speechbrain/spkrec-ecapa-voxceleb`(공개 모델, 가중치 고정, 미세조정 없음). 상용 API 없음.
- 모델 선택(C, 섞는 비율)은 Training 내부 holdout 2,920통화로만 했다(Part A 앙상블을 고른 것과 같은 holdout).

**특징 캐시**: 발화별 ECAPA 192차원 임베딩과 MFCC 통계는 Mission 2 때 Colab GPU에서
`training_code/mission2_ecapa_cache/train_speaker_embed.py`, `train_mfcc_rf.py`로 만든 캐시를 그대로 쓴다
(speaker 0/1 발화를 JSON 차례대로 잘라 32개씩 ECAPA, float16 저장). 통화별 `gender`와 파일 이름은
`export_labels_for_local.py`(JSON만 읽음)로 같은 통화 순서에 맞춰 붙였다.

In [1]:
import sys, time
from pathlib import Path
import numpy as np

SRC = Path(r"C:/유니짜장/DCC_Aideas/DCC_AIdeas/src/mission1")
M2_LOCAL = Path(r"C:/유니짜장/DCC_Aideas/mission2_local")          # Mission 2 발화별 캐시 (memmap)
EXPORT = Path(r"C:/유니짜장/DCC_Aideas/mission1_local/export_raw/mission1_export")  # 통화별 gender·stem, Part A 3모델 확률
WORK = Path(r"C:/유니짜장/DCC_Aideas/mission1_local/notebook_run")
WORK.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC))
from gender_data import load_split, yeeun_holdout_mask, pooled_features

## 1. 데이터 정렬 검증
JSON에서 센 통화별 발화 수가 캐시의 통화 경계와 **모든 통화에서** 같아야 한다(다르면 `load_split`이 멈춘다).
Part A의 내부 holdout은 stem 오름차순 + `train_test_split(test_size=0.1, random_state=42, stratify=gender)`로 재현하고,
Part A에서 저장한 split 파일과 같은지 확인한다.

In [2]:
train = load_split(M2_LOCAL, EXPORT, "train")
val = load_split(M2_LOCAL, EXPORT, "val")
for name, d in (("Train", train), ("Validation", val)):
    print(f"{name}: 통화 {len(d['counts']):,}, 발화 {int(d['counts'].sum()):,}, 신고자 발화 비율 {d['speaker'].mean():.4f}, F 비율 {d['gender'].mean():.4f}")
held = yeeun_holdout_mask(train["stems"], train["gender"], EXPORT)
print(f"내부 학습 {int((~held).sum()):,}통화 / 내부 holdout {int(held.sum()):,}통화 (F 비율 {train['gender'][held].mean():.4f})")

Train: 통화 29,200, 발화 889,358, 신고자 발화 비율 0.5197, F 비율 0.5323
Validation: 통화 3,640, 발화 111,947, 신고자 발화 비율 0.5190, F 비율 0.5382


확인: 예은 캐시 순서 = stem 오름차순
확인: 재현한 holdout = 예은 split 파일 (2,920통화)
내부 학습 26,280통화 / 내부 holdout 2,920통화 (F 비율 0.5322)


## 2. 통화 요약 특징
신고자(`speaker=1`) 발화들의 ECAPA 임베딩(길이 1로 정규화)을 발화 길이로 가중 평균한 192차원 + 표준편차 192차원 = 384차원.
발화 순서는 쓰지 않는다(평균·표준편차는 순서와 무관).

In [3]:
start = time.time()
pooled_train = pooled_features(train)
pooled_val = pooled_features(val)
np.save(WORK / "pooled_train.npy", pooled_train)
np.save(WORK / "pooled_val.npy", pooled_val)
print("특징 계산", pooled_train.shape, pooled_val.shape, f"{time.time() - start:.0f}초")

특징 계산 (29200, 1623) (3640, 1623) 22초


## 3. 로지스틱 회귀 학습
표준화(내부 학습 통화로만 fit) → 로지스틱 회귀. C는 내부 holdout 정확도(동률이면 log loss)로 선택.
Validation은 선택이 끝난 모델로 한 번만 판정한다. 비교용으로 MFCC·대원 특징을 넣은 변형도 같은 절차로 돌린다.

In [4]:
import subprocess
RUNS = WORK / "runs"
for blocks in (["caller_ecapa"], ["caller_mfcc"], ["caller_ecapa", "caller_mfcc"], ["caller_ecapa", "caller_mfcc", "agent", "counts"]):
    print("=" * 20, "+".join(blocks))
    result = subprocess.run([sys.executable, str(SRC / "train_gender_pooled.py"), "--m2_local", str(M2_LOCAL),
                             "--export_dir", str(EXPORT), "--pooled_dir", str(WORK), "--blocks", *blocks,
                             "--out", str(RUNS / ("pooled_" + "+".join(blocks)))], capture_output=True, text=True, encoding="utf-8")
    print(result.stdout.strip() or result.stderr[-2000:])

==================== caller_ecapa


확인: 예은 캐시 순서 = stem 오름차순
확인: 재현한 holdout = 예은 split 파일 (2,920통화)
C=0.003: holdout Acc 0.9815 (오류 54), logloss 0.0668
C=0.01: holdout Acc 0.9805 (오류 57), logloss 0.0665
C=0.03: holdout Acc 0.9812 (오류 55), logloss 0.0674
C=0.1: holdout Acc 0.9795 (오류 60), logloss 0.0683
C=0.3: holdout Acc 0.9791 (오류 61), logloss 0.0692
C=1.0: holdout Acc 0.9791 (오류 61), logloss 0.0698
선택 C=0.003: holdout 0.9815 / Validation 0.9802 (오류 72)
==================== caller_mfcc


확인: 예은 캐시 순서 = stem 오름차순
확인: 재현한 holdout = 예은 split 파일 (2,920통화)
C=0.003: holdout Acc 0.9527 (오류 138), logloss 0.1342
C=0.01: holdout Acc 0.9541 (오류 134), logloss 0.1274
C=0.03: holdout Acc 0.9565 (오류 127), logloss 0.1252
C=0.1: holdout Acc 0.9579 (오류 123), logloss 0.1254
C=0.3: holdout Acc 0.9565 (오류 127), logloss 0.1273
C=1.0: holdout Acc 0.9551 (오류 131), logloss 0.1296
선택 C=0.1: holdout 0.9579 / Validation 0.9541 (오류 167)
==================== caller_ecapa+caller_mfcc


확인: 예은 캐시 순서 = stem 오름차순
확인: 재현한 holdout = 예은 split 파일 (2,920통화)
C=0.003: holdout Acc 0.9818 (오류 53), logloss 0.0645
C=0.01: holdout Acc 0.9812 (오류 55), logloss 0.0651
C=0.03: holdout Acc 0.9795 (오류 60), logloss 0.0681
C=0.1: holdout Acc 0.9771 (오류 67), logloss 0.0730
C=0.3: holdout Acc 0.9771 (오류 67), logloss 0.0768
C=1.0: holdout Acc 0.9757 (오류 71), logloss 0.0827
선택 C=0.003: holdout 0.9818 / Validation 0.9802 (오류 72)
==================== caller_ecapa+caller_mfcc+agent+counts


확인: 예은 캐시 순서 = stem 오름차순
확인: 재현한 holdout = 예은 split 파일 (2,920통화)
C=0.003: holdout Acc 0.9798 (오류 59), logloss 0.0661
C=0.01: holdout Acc 0.9781 (오류 64), logloss 0.0692
C=0.03: holdout Acc 0.9760 (오류 70), logloss 0.0770
C=0.1: holdout Acc 0.9736 (오류 77), logloss 0.0911
C=0.3: holdout Acc 0.9716 (오류 83), logloss 0.1117
C=1.0: holdout Acc 0.9644 (오류 104), logloss 0.1656
선택 C=0.003: holdout 0.9798 / Validation 0.9786 (오류 78)


## 4. WavLM 3모델 앙상블과 섞는 비율 선택
Part A 3모델(40:40:20)의 내부 holdout·공식 Validation 확률(Part A에서 저장한 파일)과 같은 통화끼리 맞춰
`P(F) = (1-w)·WavLM 3모델 + w·ECAPA`의 오류를 계산한다(아래 출력의 '예은'은 Part A 3모델 앙상블). **w는 holdout 오류가 최소인 구간의 가운데로 고른다.**
Validation 열은 참고로만 출력하며 선택에 쓰지 않았다.

In [5]:
result = subprocess.run([sys.executable, str(SRC / "blend_with_yeeun.py"), "--export_dir", str(EXPORT),
                         "--runs", str(RUNS / "pooled_caller_ecapa.npz")], capture_output=True, text=True, encoding="utf-8")
print(result.stdout.strip() or result.stderr[-2000:])

[holdout] 통화 2920: 예은 앙상블 오류 61, 예은 개별 [66, 65, 71], 새 모델 오류 54 (개별 [54])
   겹침: 둘 다 오류 34, 예은만 27, 새 모델만 20
[val] 통화 3640: 예은 앙상블 오류 77, 예은 개별 [88, 86, 87], 새 모델 오류 72 (개별 [72])
   겹침: 둘 다 오류 37, 예은만 40, 새 모델만 35

새 모델 비율 w: (1-w)·예은 + w·새 모델
 w     holdout오류  Val오류
 0.00      61          77
 0.05      60          76
 0.10      59          76
 0.15      57          73
 0.20      56          72
 0.25      54          73
 0.30      57          71
 0.35      58          72
 0.40      59          67
 0.45      59          68
 0.50      59          69
 0.55      54          67
 0.60      51          65
 0.65      51          65
 0.70      49          64
 0.75      49          63
 0.80      50          63
 0.85      51          65
 0.90      54          68
 0.95      55          70
 1.00      54          72

holdout으로 고른 w=0.75: holdout 오류 49 (예은 61), Validation 오류 63 (예은 77) → -14개, Val Acc 0.9827


## 5. 제출용 가중치와 재현 확인
제출 폴더의 `ecapa_gender/ecapa_lr.pt`에는 표준화 평균·표준편차, 로지스틱 회귀 계수·절편과 섞는 비율(0.75)만
텐서·숫자로 들어 있다(sklearn 버전과 무관). 제출 코드(`ecapa_gender.py`)의 특징·확률 계산이 학습 때와 같은지 확인한다.

In [6]:
SUB = Path(r"C:/유니짜장/DCC_Aideas/mission1_submission_final")
sys.path.insert(0, str(SUB))
import torch
from ecapa_gender import EcapaGender, caller_feature, load_lr_weights
model = EcapaGender.__new__(EcapaGender)   # 인코더 없이 가중치만 불러 계산 확인
model.mean, model.scale, model.coef, model.intercept, model.blend = load_lr_weights(SUB / "ecapa_gender" / "ecapa_lr.pt")
print("C =", torch.load(SUB / "ecapa_gender" / "ecapa_lr.pt", weights_only=True)["C"], "/ ECAPA 비율 =", model.blend)
trained = np.load(RUNS / "pooled_caller_ecapa.npz")
probs = []
for c in range(len(val["counts"])):
    span = slice(val["offsets"][c], val["offsets"][c + 1]); caller = val["speaker"][span] == 1
    feature = caller_feature(np.asarray(val["ecapa"][span], np.float32)[caller], np.asarray(val["mfcc"][span][:, -1], np.float32)[caller])
    probs.append(model.probability_from_feature(feature))
probs = np.array(probs)
print("제출 코드 확률 vs 학습 확률 최대 차이:", np.abs(probs - trained["val_prob"]).max(),
      "/ 판정이 다른 통화:", int(((probs >= .5) != (trained["val_prob"] >= .5)).sum()))

C = 0.003 / ECAPA 비율 = 0.75


제출 코드 확률 vs 학습 확률 최대 차이: 1.4843632278882524e-07 / 판정이 다른 통화: 0


## 6. 실데이터 추론 검증 (Colab T4)
최종 제출 폴더를 공식 명령 그대로 Validation WAV 3,640개에 실행한 결과(출력 CSV와 파일별 확률)를 채점한다.
WavLM 3모델 확률이 Part A에서 저장한 값과, ECAPA 확률이 위 학습 값과 같은지도 비교한다.
(이 실행 뒤 추론 코드에서 쓰지 않는 함수만 정리하고 가중치 파일 형식을 .pt로 바꿨으며, 정리 전후 출력이 같음을 로컬에서 확인했다.)

In [7]:
import csv
RESULT = Path(r"C:/유니짜장/DCC_Aideas/mission1_local/colab_full_result")
with open(RESULT / "mission1.csv", encoding="utf-8") as f:
    rows = list(csv.reader(f))
truth = dict(zip(val["stems"], val["gender"]))
pred = {Path(n).stem: int(g == "F") for n, g in rows[1:]}
wrong = sum(pred[s] != truth[s] for s in truth)
print("헤더", rows[0], "/ 행", len(rows) - 1)
print(f"공식 명령 재추론 Validation 정확도 {1 - wrong / len(truth):.4f} (오류 {wrong}/{len(truth)})")
ref = np.load(Path(r"C:/유니짜장/DCC_Aideas/mission1_local/colab_v2/verify/v2_reference_val.npz"))
got = np.load(RESULT / "probs.npz")
index = np.array([{str(s): i for i, s in enumerate(ref["stems"])}[Path(str(n)).stem] for n in got["names"]])
w = np.array([.4, .4, .2])
print("WavLM 3모델 확률 최대 차이:", np.abs(got["yeeun"] - ref["yeeun"][index]).max(axis=0),
      "/ WavLM 3모델 앙상블 단독 오류:", int(((got["yeeun"] @ w >= .5) != ref["y"][index]).sum()))
print("ECAPA 확률 최대 차이:", float(np.abs(got["ecapa"] - ref["ecapa"][index]).max()),
      "/ ECAPA 단독 오류:", int(((got["ecapa"] >= .5) != ref["y"][index]).sum()))
print("최종 판정이 로컬 계산과 다른 통화:", int(((got["final"] >= .5) != (ref["final"][index] >= .5)).sum()))

헤더 ['audio file name', 'gender'] / 행 3640
공식 명령 재추론 Validation 정확도 0.9827 (오류 63/3640)


WavLM 3모델 확률 최대 차이: [9.47713852e-06 1.29938126e-05 2.63452530e-05] / WavLM 3모델 앙상블 단독 오류: 77
ECAPA 확률 최대 차이: 2.0679528823286386e-05 / ECAPA 단독 오류: 72
최종 판정이 로컬 계산과 다른 통화: 0


## 결과 요약

| 모델 | 내부 holdout 오류 (2,920) | Validation 오류 (3,640) | Validation Accuracy |
| --- | ---: | ---: | ---: |
| WavLM 3모델 40:40:20 (Part A) | 61 | 77 | 0.9788 |
| ECAPA + 로지스틱 회귀 (Part B) | 54 | 72 | 0.9802 |
| **최종: WavLM 3모델 25% + ECAPA 75%** | **49** | **63** | **0.9827** |

두 모델은 서로 다른 통화에서 틀린다(Validation: 둘 다 오류 37, WavLM만 40, ECAPA만 35). 그래서 섞으면 오류가 줄어든다.
통화 단위 트랜스포머(신고자·대원 발화 전체를 넣는 모델)도 시험했지만 holdout 오류 59개로 이득이 없어 쓰지 않았다.

# 최종 추론 및 제출

```bash
python inference.py --audio_dir {wav folder} --label_dir {json folder} \
    --ckpt_path ./ecapa_gender/ecapa_lr.pt --output ./outputs/mission1.csv
```

- `inference.py` → `inference_ecapa_blend.py`: 통화마다 WavLM 3모델 확률과 ECAPA 확률을 구해 25:75로 섞는다.
- 모델 코드: `wavlm_3model.py`(WavLM 3모델), `common.py`(신고자 음성·F0), `ecapa_gender.py`(ECAPA 특징·로지스틱 회귀).
- 가중치: `attention_f0/`, `stats_f0/`, `stats_no_f0/`(.pt), `ecapa_gender/ecapa_lr.pt`, 사전학습 모델 `pretrained/`, `ecapa_gender/encoder/`.
- 출력: `audio file name,gender` CSV (통화당 한 행, M/F).

| 모델 | 내부 holdout 오류 (2,920) | Validation 오류 (3,640) | Validation Accuracy |
| --- | ---: | ---: | ---: |
| Log-Mel CNN (기준선) | – | 220 | 0.9396 |
| MFCC+F0 RandomForest (기준선) | – | 210 | 0.9423 |
| WavLM 3모델 앙상블 (Part A) | 61 | 77 | 0.9788 |
| ECAPA 로지스틱 회귀 (Part B) | 54 | 72 | 0.9802 |
| **최종 (25:75)** | **49** | **63** | **0.9827** |